# dist-send-recv-pair — faded example 1: Allocate a zeros_like receive buffer before calling dist.recv

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`. Running the beacon reports progress on the `Distributed: dist.send/recv pair` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Before calling `dist.recv`, you must allocate a buffer tensor with the correct shape and dtype to receive the incoming data. The standard pattern is `buf = torch.zeros_like(reference_tensor)` or `buf = torch.zeros(size, dtype=dtype)`. The `dist.recv` call writes into this buffer in-place; the buffer is your only handle to the received data.

## Faded exercise 1

Complete `recv_into_buffer`. The sender's payload size is known. Allocate a float32 zero buffer of that size and receive into it.

**Fill in:** Allocate `recv_buf` as a float32 zeros tensor of length `payload_size`, then call `dist.recv(recv_buf, src=src)` to fill it.

In [ ]:
def recv_into_buffer(rank, world_size, dist_module, src, payload_size, send_tensor=None):
    """rank != src: allocate buffer and receive. rank == src: send."""
    if rank == src:
        dist_module.send(send_tensor, dst=(src + 1) % world_size)
        return None
    else:
        raise NotImplementedError()  # TODO: Allocate `recv_buf` as a float32 zeros tensor of length `payload_size`, then call `dist.recv(recv_buf, src=src)` to fill it.
        return recv_buf


def _test():
    import torch
    from unittest.mock import MagicMock, call

    recv_data = torch.tensor([7.0, 8.0, 9.0])

    def mock_recv(buf, src):
        buf.copy_(recv_data)

    dist_mock = MagicMock()
    dist_mock.recv.side_effect = mock_recv

    result = recv_into_buffer(rank=1, world_size=2, dist_module=dist_mock,
                               src=0, payload_size=3)
    assert result is not None
    assert result.shape == (3,)
    assert result.dtype == torch.float32
    assert torch.allclose(result, recv_data)
    dist_mock.recv.assert_called_once()


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def recv_into_buffer(rank, world_size, dist_module, src, payload_size, send_tensor=None):
    """rank != src: allocate buffer and receive. rank == src: send."""
    if rank == src:
        dist_module.send(send_tensor, dst=(src + 1) % world_size)
        return None
    else:
        import torch
        recv_buf = torch.zeros(payload_size, dtype=torch.float32)
        dist_module.recv(recv_buf, src=src)
        return recv_buf
```
</details>